In [1]:
# Pathway enrtichment pipeline 

import warnings
import os
import time
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path
import gseapy as gp

warnings.filterwarnings("ignore")
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt.rcParams["figure.dpi"] = 300
sc.settings.set_figure_params(dpi=300, dpi_save=300)


# Configurations
# edgeR input sources
EDGER_BASE = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1"
EDGER_SOURCES = [
    (os.path.join(EDGER_BASE, "cell_type/Ref_control/results/edgeR_significant"),  "cell_type",  "Ref_control"),
    (os.path.join(EDGER_BASE, "cell_type/Ref_3month/results/edgeR_significant"),   "cell_type",  "Ref_3month"),
    (os.path.join(EDGER_BASE, "cell_state/Ref_control/results/edgeR_significant"), "cell_state", "Ref_control"),
    (os.path.join(EDGER_BASE, "cell_state/Ref_3month/results/edgeR_significant"),  "cell_state", "Ref_3month"),
]

# Output base directory
OUT_BASE = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2"

# --- C. Species prefixes to strip ---
PREFIXES = ["Sus_scrofa___", "Papio_anubis_", "pCMV_____"]

# Species-origin mapping for cross-species duplicate resolution
#   Cell types originating from Sus scrofa (pig donor heart)
SUS_CELLTYPES   = {"CM", "EC", "FB", "Mye_sus", "Lymph_sus", "NC", "MC"}
#   Cell types originating from Papio anubis (baboon recipient)
PAPIO_CELLTYPES = {"Lymph_papio", "Mye_papio"}

# Gene sets for enrichment 
GENE_SETS = [
    "MSigDB_Hallmark_2020",
]

# Enrichment thresholds
ENRICHMENT_FDR_CUTOFF = 0.05

# Rate-limiting parameters 
API_DELAY_SECONDS = 1.5
API_MAX_RETRIES   = 3
API_RETRY_WAIT    = 10

# Visualization params
BAR_WIDTH          = 0.6
BAR_EDGE_COLOR     = "black"
BAR_EDGE_WIDTH     = 0.5
BAR_ALPHA          = 0.96
DOT_SIZE_DEFAULT   = 100
DOT_ALPHA          = 0.98
UP_COLOR           = "salmon"
DOWN_COLOR         = "skyblue"
MAX_PATHWAYS_PLOT  = 40
TOP_N_BUBBLE       = 5
BUBBLE_MIN_SIZE    = 30
BUBBLE_MAX_SIZE    = 150


# Section 2: Helper functions

In [2]:
# Helper functions

def remove_prefix(gene: str) -> str:
    """Strip species prefix from a gene name."""
    for prefix in PREFIXES:
        if gene.startswith(prefix):
            return gene[len(prefix):]
    return gene


def get_species(gene: str) -> str:
    """Return the species prefix a gene belongs to, or 'unknown'."""
    if gene.startswith("Sus_scrofa___"):
        return "Sus_scrofa___"
    elif gene.startswith("Papio_anubis_"):
        return "Papio_anubis_"
    elif gene.startswith("pCMV_____"):
        return "pCMV_____"
    return "unknown"


def get_preferred_prefix(celltype: str) -> str:
    """
    Given a cell type name, return the species prefix to KEEP when a gene
    appears under both species.

    For cell_state level: celltype is the parent cell_type folder name from
    the directory structure (e.g. 'CM', 'Lymph_papio'), so the mapping works.
    """
    if celltype in SUS_CELLTYPES:
        return "Sus_scrofa___"
    elif celltype in PAPIO_CELLTYPES:
        return "Papio_anubis_"
    else:
        print(f"    [WARN] Cell type '{celltype}' not in SUS_CELLTYPES or "
              f"PAPIO_CELLTYPES. Defaulting to Sus_scrofa___.")
        return "Sus_scrofa___"


def resolve_species_duplicates(df: pd.DataFrame, celltype: str) -> pd.DataFrame:
    """
    Resolve cross-species duplicate genes and strip prefixes.

    Logic:
      1. Parse each gene's species and bare name.
      2. Find bare names that appear under BOTH Sus_scrofa___ and Papio_anubis_.
      3. For duplicates: keep only the row(s) from the preferred species
         (determined by cell type origin).
      4. For genes unique to one species: keep as-is.
      5. Strip all prefixes from the Gene column.

    Returns a DataFrame with deduplicated, prefix-free Gene names.
    """
    df = df.copy()
    df["_species"] = df["Gene"].astype(str).apply(get_species)
    df["_bare"]    = df["Gene"].astype(str).apply(remove_prefix)

    # Identify bare gene names under both Sus_scrofa___ and Papio_anubis_
    sus_bare   = set(df.loc[df["_species"] == "Sus_scrofa___",   "_bare"])
    papio_bare = set(df.loc[df["_species"] == "Papio_anubis_", "_bare"])
    shared_bare = sus_bare & papio_bare

    n_shared = len(shared_bare)

    if n_shared > 0:
        preferred = get_preferred_prefix(celltype)
        is_shared    = df["_bare"].isin(shared_bare)
        is_preferred = df["_species"] == preferred
        # Keep: (not a duplicate gene) or  (duplicate gene and preferred species)
        mask = (~is_shared) | (is_shared & is_preferred)
        n_before  = len(df)
        df = df[mask].copy()
        n_dropped = n_before - len(df)
        pref_short = preferred.replace("_", " ").strip()
        print(f"    [DEDUP] {n_shared} gene(s) under both species. "
              f"Kept {pref_short} versions. Dropped {n_dropped} row(s).")
    else:
        print(f"    [DEDUP] No cross-species duplicates.")

    # Strip all prefixes
    df["Gene"] = df["_bare"]
    df.drop(columns=["_species", "_bare"], inplace=True)
    return df


def collect_sig_csvs(sig_dir: str, level: str):
    """
    Walk the edgeR_significant directory and yield tuples of:
      (csv_path, comparison, celltype, cellstate_or_None)

    cell_type level:  sig_dir/<comparison>/<celltype>.csv
    cell_state level: sig_dir/<comparison>/<celltype>/<cellstate>.csv
    """
    if not os.path.isdir(sig_dir):
        print(f"  [SKIP] Directory not found: {sig_dir}")
        return

    for comp_folder in sorted(os.listdir(sig_dir)):
        comp_path = os.path.join(sig_dir, comp_folder)
        if not os.path.isdir(comp_path):
            continue

        if level == "cell_type":
            for fname in sorted(os.listdir(comp_path)):
                if not fname.endswith(".csv"):
                    continue
                csv_path = os.path.join(comp_path, fname)
                celltype = (fname.replace(".csv", "")
                            .replace("edgeR_Sig_", "")
                            .split(f"_{comp_folder}")[0])
                yield csv_path, comp_folder, celltype, None

        elif level == "cell_state":
            for ct_folder in sorted(os.listdir(comp_path)):
                ct_path = os.path.join(comp_path, ct_folder)
                if not os.path.isdir(ct_path):
                    continue
                for fname in sorted(os.listdir(ct_path)):
                    if not fname.endswith(".csv"):
                        continue
                    csv_path = os.path.join(ct_path, fname)
                    cellstate = (fname.replace(".csv", "")
                                 .replace("edgeR_Sig_", "")
                                 .split(f"_{comp_folder}")[0])
                    yield csv_path, comp_folder, ct_folder, cellstate


def build_output_dir(level: str, ref_label: str, comparison: str,
                     celltype: str = None, cellstate: str = None) -> str:
    """Build output directory following the standard structure."""
    parts = [OUT_BASE, level, ref_label, comparison]
    if celltype:
        parts.append(celltype)
    if cellstate:
        parts.append(cellstate)
    out = os.path.join(*parts)
    os.makedirs(out, exist_ok=True)
    return out


def run_enrichr_direction(gene_list: list, gene_set: str,
                          direction: str) -> pd.DataFrame:
    """Run gseapy enrichr for a gene list and tag with direction.
    Includes retry logic for API rate limiting (HTTP 429)."""
    if not gene_list:
        return pd.DataFrame()

    for attempt in range(1, API_MAX_RETRIES + 1):
        try:
            time.sleep(API_DELAY_SECONDS)
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=gene_set,
                organism="human",
                outdir=None,
                cutoff=0.5,
                no_plot=True,
            )
            break
        except Exception as e:
            if "429" in str(e) and attempt < API_MAX_RETRIES:
                print(f"    [RATE-LIMITED] {direction} -- waiting "
                      f"{API_RETRY_WAIT}s (attempt {attempt}/{API_MAX_RETRIES})")
                time.sleep(API_RETRY_WAIT)
            else:
                print(f"    [WARN] enrichr failed for {direction}: {e}")
                return pd.DataFrame()

    if enr.results is None or enr.results.empty:
        return pd.DataFrame()

    res = enr.results.copy()
    pval_col = ("Adjusted P-value" if "Adjusted P-value" in res.columns
                else "FDR")
    res = res[res[pval_col] < ENRICHMENT_FDR_CUTOFF]

    if res.empty:
        return pd.DataFrame()

    res["Regulation"] = direction
    return res



# Section 3: Species-aware prefix elimination & pathway enrichment

In [5]:

print("=" * 70)
print("STEP 1: Species-aware duplicate resolution + Prefix removal + Enrichment")
print("=" * 70)

all_enrichment_records = []

for sig_dir, level, ref_label in EDGER_SOURCES:
    print(f"\n--- {level} / {ref_label} ---")
    print(f"    Source: {sig_dir}")

    for csv_path, comparison, celltype, cellstate in collect_sig_csvs(sig_dir, level):
        label = f"{celltype}/{cellstate}" if cellstate else celltype
        print(f"  [{comparison}] {label}")

        # Read edgeR significant CSV
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"    [ERROR] {e}")
            continue
        if "Gene" not in df.columns:
            print(f"    [SKIP] No 'Gene' column")
            continue

        # First resolve cross-species duplicates then strip prefixes.
        # For cell_state level, 'celltype' is already the parent cell_type
        # folder (e.g. 'CM', 'Lymph_papio'), so the mapping works directly.
        df = resolve_species_duplicates(df, celltype)
        

        # Build output directory
        out_dir = build_output_dir(level, ref_label, comparison,
                                   celltype, cellstate)

        # Save prefix-free, deduplicated CSV
        prefix_free_csv = os.path.join(out_dir,
                                       "prefix_free_significant_DEGs.csv")
        df.to_csv(prefix_free_csv, index=False)

        # Split up/down genes
        if "type" in df.columns:
            up_genes   = (df[df["type"] == "up"]["Gene"]
                          .dropna().unique().tolist())
            down_genes = (df[df["type"] == "down"]["Gene"]
                          .dropna().unique().tolist())
        elif "logFC" in df.columns:
            up_genes   = (df[df["logFC"] > 0]["Gene"]
                          .dropna().unique().tolist())
            down_genes = (df[df["logFC"] < 0]["Gene"]
                          .dropna().unique().tolist())
        else:
            print(f"    [SKIP] Cannot determine direction "
                  f"(no 'type' or 'logFC' column)")
            continue

        # Run enrichment per gene set
        for gs in GENE_SETS:
            gs_dir = os.path.join(out_dir, gs)
            os.makedirs(gs_dir, exist_ok=True)

            res_up   = run_enrichr_direction(up_genes,   gs, "Up-regulated")
            res_down = run_enrichr_direction(down_genes, gs, "Down-regulated")
            all_pathways = pd.concat([res_up, res_down], ignore_index=True)

            if all_pathways.empty:
                continue

            # Save enrichment CSV
            enr_csv = os.path.join(gs_dir, "significant_enrichment.csv")
            all_pathways.to_csv(enr_csv, index=False)
            print(f"    -> {gs}: {len(res_up)} up, "
                  f"{len(res_down)} down pathways saved")

            # Tag for downstream plots
            all_pathways["CellType"]   = celltype
            all_pathways["CellState"]  = cellstate if cellstate else "NA"
            all_pathways["Comparison"] = comparison
            all_pathways["Level"]      = level
            all_pathways["Ref"]        = ref_label
            all_pathways["GeneSet"]    = gs
            all_enrichment_records.append(all_pathways)

print("\nStep 1 complete.")


STEP 1: Species-aware duplicate resolution + Prefix removal + Enrichment

--- cell_type / Ref_control ---
    Source: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_control/results/edgeR_significant
  onth_protocol_vs_Control] CM
    [DEDUP] 142 gene(s) under both species. Kept Sus scrofa versions. Dropped 142 row(s).
    -> MSigDB_Hallmark_2020: 4 up, 6 down pathways saved
  onth_protocol_vs_Control] EC
    [DEDUP] 2 gene(s) under both species. Kept Sus scrofa versions. Dropped 2 row(s).
    -> MSigDB_Hallmark_2020: 23 up, 5 down pathways saved
  onth_protocol_vs_Control] FB
    [DEDUP] 1 gene(s) under both species. Kept Sus scrofa versions. Dropped 1 row(s).
    -> MSigDB_Hallmark_2020: 21 up, 3 down pathways saved
  onth_protocol_vs_Control] MC
    [DEDUP] No cross-species duplicates.
    -> MSigDB_Hallmark_2020: 9 up, 0 down pathways saved
  onth_protocol_vs_Control] Mye_sus
    [DEDUP] No cross-species duplicates.
    -> MSigDB_Hallma

# Top 10 mirrored barplots

In [9]:
# standalone mirrored barplots and dotplots 

import os
import re
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import warnings
warnings.filterwarnings("ignore")
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42
plt.rcParams["figure.dpi"]   = 300

# Configs

# Base enrichment path
OUT_BASE   = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2"

# Specific experiment targeting
LEVEL      = "cell_type"                           # 'cell_type' or 'cell_state'
REF_LABEL  = "Ref_3month"                          # e.g., 'Ref_control' or 'Ref_3month'

# Enter the exact conditions as they appear in the metadata (with dashes, slashes, etc.)
TEST_COND  = "Rejection"
REF_COND   = "3month-protocol"

CELL_TYPE  = "Mye_papio"                                   # Target cell type
GENESET    = "MSigDB_Hallmark_2020"                # Target gene set

# --- Pathway Filtering Modes ---
# MODE 1: Top N overall (Defaults to Top 10 lowest P-values regardless of direction)
# TOP_N_OVERALL = 10

# MODE 2 (Alternative): Uncomment the variables below (and the logic in block 3) 
# to instead show Top N Up-regulated AND Top N Down-regulated separately.
TOP_N_UP   = 5   # Number of top up-regulated pathways to show
TOP_N_DOWN = 5   # Number of top down-regulated pathways to show

# Visualization parameters

BAR_WIDTH        = 0.7
BAR_EDGE_COLOR   = "black"
BAR_EDGE_WIDTH   = 0.5
BAR_ALPHA        = 0.96
UP_COLOR         = "salmon"
DOWN_COLOR       = "skyblue"

MIRRORED_PLOT_WIDTH      = 6
MIRRORED_MIN_HEIGHT      = 4.0
MIRRORED_HEIGHT_PER_TERM = 0.1

LEGEND_HANDLES = [
    Line2D([0], [0], marker="s", color="w", label="Up-regulated",
           markerfacecolor=UP_COLOR, markersize=12, markeredgecolor="k"),
    Line2D([0], [0], marker="s", color="w", label="Down-regulated",
           markerfacecolor=DOWN_COLOR, markersize=12, markeredgecolor="k"),
]

# Data loading and filtering

# # Replicate the edgeR R script sanitization logic: gsub("[^A-Za-z0-9_]", "_", cond)
# test_cond_safe = re.sub(r'[^A-Za-z0-9_]', '_', TEST_COND)

# # The reference condition was appended as-is in your edgeR script
# comparison_folder = f"{test_cond_safe}_vs_{REF_COND}"

# # Construct the exact path where the enrichment CSV is stored
# target_dir = os.path.join(OUT_BASE, LEVEL, REF_LABEL, comparison_folder, CELL_TYPE, GENESET)
# csv_path   = os.path.join(target_dir, "significant_enrichment.csv")

# if not os.path.exists(csv_path):
#     raise FileNotFoundError(f"Cannot find enrichment CSV at:\n{csv_path}\nPlease check your configurations.")

# df = pd.read_csv(csv_path)

# if df.empty or "Regulation" not in df.columns:
#     raise ValueError("The enrichment CSV is empty or missing the 'Regulation' column.")

# pval_col  = "Adjusted P-value" if "Adjusted P-value" in df.columns else "FDR"
# score_col = "Combined Score" if "Combined Score" in df.columns else "Odds Ratio"

# ------------------------------------------------------------------------------
# APPLY FILTERING MODE
# ------------------------------------------------------------------------------

# MODE 1: Top N Overall -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# df_plot = df.sort_values(pval_col).head(TOP_N_OVERALL).copy()

# MODE 2: Top N Per Direction (Currently Active)
# To use Mode 1 instead, uncomment the Mode 1 line above, and comment the 3 lines below:
df_up   = df[df["Regulation"] == "Up-regulated"].sort_values(pval_col).head(TOP_N_UP)
df_down = df[df["Regulation"] == "Down-regulated"].sort_values(pval_col).head(TOP_N_DOWN)
df_plot = pd.concat([df_up, df_down]).copy()

# ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

if df_plot.empty:
    raise ValueError("No pathways matched the filtering criteria.")

# ==============================================================================
# 4. Preparing for plottings
# ==============================================================================

# Map colors and mirroring signs
df_plot["Color"] = df_plot["Regulation"].map({"Up-regulated": UP_COLOR, "Down-regulated": DOWN_COLOR})
df_plot["Sign"]  = df_plot["Regulation"].map({"Up-regulated": 1, "Down-regulated": -1})
df_plot["Mirrored_Score"] = df_plot[score_col] * df_plot["Sign"]

# Sort Upregulated first, then Downregulated for visual grouping
df_plot = df_plot.sort_values(by=["Sign", score_col], ascending=[False, False])

# Dynamic margin and height layout calculations
max_label_len = df_plot["Term"].astype(str).str.len().max()
left_margin   = min(0.5, max(0.27, 0.27 + 0.0037 * max_label_len ** 1.35))
fig_height    = max(MIRRORED_MIN_HEIGHT, MIRRORED_HEIGHT_PER_TERM * len(df_plot))

# Create and save the plots

fig, ax = plt.subplots(figsize=(MIRRORED_PLOT_WIDTH, fig_height), constrained_layout=False)
ax.grid(False)

# Draw Bars
ax.barh(
    y=df_plot["Term"], 
    width=df_plot["Mirrored_Score"], 
    color=df_plot["Color"],
    edgecolor=BAR_EDGE_COLOR, 
    linewidth=BAR_EDGE_WIDTH,
    alpha=BAR_ALPHA, 
    height=BAR_WIDTH
)

# Center line
ax.axvline(0, color="black", linestyle="--", linewidth=1)

# Formatting
ax.set_xlabel(f"{score_col} (Up = Right, Down = Left)", fontsize=8, fontweight="bold")
ax.set_title(f"Targeted Mirrored Barplot: {CELL_TYPE} ({TEST_COND} vs {REF_COND})\n{GENESET}", fontsize=10, fontweight="bold", pad=12)
ax.invert_yaxis()
ax.margins(y=0.01)

for spine in ["top", "bottom", "left", "right"]:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_linewidth(1.0)
    ax.spines[spine].set_color("black")

# Legend
ax.legend(
    handles=LEGEND_HANDLES, 
    title="Regulation",
    loc="center left", 
    bbox_to_anchor=(1.01, 0.5),
    fontsize=9, 
    title_fontproperties={"weight": "bold"},
    frameon=True
)

plt.subplots_adjust(left=left_margin, right=0.80, top=0.90, bottom=0.1)

# Save
plot_out_name = f"Targeted_Barplot_{CELL_TYPE}_{comparison_folder}_Top{len(df_plot)}.pdf"
plot_out_path = os.path.join(target_dir, plot_out_name)

plt.savefig(plot_out_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"\n✓ Successfully saved targeted plot ({len(df_plot)} pathways) to:")
print(f"  {plot_out_path}")



✓ Successfully saved targeted plot (7 pathways) to:
  /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_type/Ref_3month/Rejection_vs_3month-protocol/Mye_papio/MSigDB_Hallmark_2020/Targeted_Barplot_Mye_papio_Rejection_vs_3month-protocol_Top7.pdf


# Bubble plots of shared pathways

In [5]:

import os
import pandas as pd
from pathlib import Path

# Base directory where Step 1 saved all the results
OUT_BASE = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2"

print("=" * 70)
print("RECONSTRUCTING ENRICHMENT RECORDS FROM DISK")
print("=" * 70)

all_enrichment_records = []
csv_count = 0

# Recursively find all saved significant_enrichment.csv files
for csv_path in Path(OUT_BASE).rglob("significant_enrichment.csv"):
    
    # Extract the directory structure relative to the OUT_BASE
    # Expected structure for cell_type:  Level / Ref / Comparison / CellType / GeneSet / file.csv (6 parts)
    # Expected structure for cell_state: Level / Ref / Comparison / CellType / CellState / GeneSet / file.csv (7 parts)
    rel_parts = csv_path.relative_to(OUT_BASE).parts
    
    if len(rel_parts) == 6:
        level, ref_label, comp, ctype, gs, fname = rel_parts
        cstate = "NA"
    elif len(rel_parts) == 7:
        level, ref_label, comp, ctype, cstate, gs, fname = rel_parts
    else:
        # Skip if it doesn't match our exact folder depth structure
        continue
        
    try:
        df = pd.read_csv(csv_path)
        
        # Only process if the CSV actually has data
        if not df.empty:
            # Re-attach the exact metadata columns required by the bubble plot engine
            df["CellType"]   = ctype
            df["CellState"]  = cstate
            df["Comparison"] = comp
            df["Level"]      = level
            df["Ref"]        = ref_label
            df["GeneSet"]    = gs
            
            all_enrichment_records.append(df)
            csv_count += 1
            
    except Exception as e:
        print(f"[ERROR] Could not read {csv_path}: {e}")

print(f"✓ Successfully loaded and reconstructed {csv_count} enrichment records.")
print("You may now run the STEP 3 Bubble Plot code block!")


RECONSTRUCTING ENRICHMENT RECORDS FROM DISK
✓ Successfully loaded and reconstructed 273 enrichment records.
You may now run the STEP 3 Bubble Plot code block!


In [7]:
# Cross-celltype bubbleplots
print("\n" + "=" * 70)
print("STEP 3: Cross-cell-type bubble plots")
print("=" * 70)

# Bubbleplot configs
BUBBLE_WIDTH_PER_CT    = 0.01
BUBBLE_HEIGHT_PER_TERM = 0.01
BUBBLE_BASE_WIDTH      = 5
BUBBLE_BASE_HEIGHT     = 3
MIN_OVERLAP            = 3

if not all_enrichment_records:
    print("[SKIP] No enrichment records to plot.")
else:
    master_df = pd.concat(all_enrichment_records, ignore_index=True)
    pval_col = ("Adjusted P-value" if "Adjusted P-value" in master_df.columns
                else "FDR")

    def scale_sizes(values, vmin, vmax, size_min, size_max):
        if vmax == vmin:
            return np.full(len(values), (size_min + size_max) / 2)
        normalized = (values - vmin) / (vmax - vmin)
        return size_min + normalized * (size_max - size_min)

    for (level, ref_label, comparison, gs), grp in master_df.groupby(
        ["Level", "Ref", "Comparison", "GeneSet"]
    ):
        if grp.empty:
            continue

        grp = grp.copy()
        
        # 1. Calculate Standard -log10(FDR) and assign mathematical sign
        grp["-log10(FDR)"] = -np.log10(grp[pval_col].astype(float).clip(lower=1e-300))
        grp["Sign"] = grp["Regulation"].map({"Up-regulated": 1, "Down-regulated": -1})
        
        # 2. Calculate Signed Significance (+ for Up, - for Down)
        grp["Signed_Sig"] = grp["-log10(FDR)"] * grp["Sign"]

        # ------------------------------------------------
        # Resolve opposing pathways in the same cell type
        # ------------------------------------------------
        # Sum the signed significance. If Up = 5 and Down = -2, Net = 3 (Up).
        net_grp = grp.groupby(["CellType", "Term"], as_index=False).agg(
            Net_Signed_Sig=("Signed_Sig", "sum")
        )
        
        # Drop instances where opposing signals perfectly cancel out to zero
        net_grp = net_grp[net_grp["Net_Signed_Sig"] != 0].copy()
        
        # Re-derive absolute significance and categorical regulation from the Net
        net_grp["-log10(FDR)"] = net_grp["Net_Signed_Sig"].abs()
        net_grp["Regulation"]  = np.where(net_grp["Net_Signed_Sig"] > 0, "Up-regulated", "Down-regulated")
        net_grp["Sign"]        = np.where(net_grp["Net_Signed_Sig"] > 0, 1, -1)
        # ---------------------------------------------

        # Overlapping pathways (shared by >= MIN_OVERLAP cell types)
        term_overlap = net_grp.groupby("Term")["CellType"].nunique()
        overlapping  = term_overlap[term_overlap >= MIN_OVERLAP]
        if overlapping.empty:
            continue

        # Rank: most shared first, then by mean significance
        term_sig = net_grp.groupby("Term")["-log10(FDR)"].mean()
        rank_df = (
            pd.DataFrame({"Term": overlapping.index,
                           "n_ct": overlapping.values})
            .merge(term_sig.rename("avg_sig"),
                   left_on="Term", right_index=True)
            .sort_values(["n_ct", "avg_sig"], ascending=[False, False])
        )
        
        top_terms = rank_df["Term"].head(TOP_N_BUBBLE).tolist()
        plot_df = net_grp[net_grp["Term"].isin(top_terms)].copy()
        
        if plot_df.empty:
            continue

        # Sort: Up-regulated on top, Down on bottom
        term_stats = (
            plot_df.groupby("Term")
            .agg(mean_sign=("Sign", "mean"),
                 mean_sig=("-log10(FDR)", "mean"))
            .sort_values(["mean_sign", "mean_sig"], ascending=[True, True])
        )
        sorted_terms = term_stats.index.tolist()
        plot_df["Term"] = pd.Categorical(
            plot_df["Term"], categories=sorted_terms, ordered=True)
        plot_df = plot_df.sort_values(["Term", "CellType"])

        # Dot colors
        plot_df["DotColor"] = plot_df["Regulation"].map(
            {"Up-regulated": UP_COLOR, "Down-regulated": DOWN_COLOR}
        ).fillna("lightgray")

        # Dot sizes based on the NET significance
        neg_log_fdr = plot_df["-log10(FDR)"].values
        dot_sizes = scale_sizes(neg_log_fdr,
                                vmin=neg_log_fdr.min(),
                                vmax=neg_log_fdr.max(),
                                size_min=BUBBLE_MIN_SIZE,
                                size_max=BUBBLE_MAX_SIZE)

        # Figure dimensions
        num_ct    = plot_df["CellType"].nunique()
        num_terms = len(sorted_terms)
        fig_w = BUBBLE_BASE_WIDTH  + num_ct    * BUBBLE_WIDTH_PER_CT
        fig_h = BUBBLE_BASE_HEIGHT + num_terms * BUBBLE_HEIGHT_PER_TERM

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))

        ax.scatter(plot_df["CellType"], plot_df["Term"],
                   s=dot_sizes, c=plot_df["DotColor"],
                   edgecolor="black", linewidth=0.5,
                   alpha=0.93, zorder=3)

        ax.grid(True, linestyle="--", linewidth=0.5,
                color="gray", alpha=0.5, zorder=0)
        ax.margins(x=0.15, y=0.15)
        ax.set_xlabel("Cell Type", fontsize=7, fontweight="bold")
        ax.set_ylabel("")
        ax.set_title(
            f"Bubble plot: {comparison.replace('_', ' ')}\n"
            f"(Top {TOP_N_BUBBLE} overlapping, {gs})",
            fontsize=8, fontweight="bold", pad=8)
        plt.xticks(rotation=90, ha="center", fontsize=6, fontweight="bold")
        plt.yticks(fontsize=7, fontweight="bold")

        # Regulation legend
        reg_legend = ax.legend(
            handles=[
                Line2D([0], [0], marker="o", color="w", label="Up-regulated",
                       markerfacecolor=UP_COLOR, markeredgecolor="k",
                       markersize=7),
                Line2D([0], [0], marker="o", color="w", label="Down-regulated",
                       markerfacecolor=DOWN_COLOR, markeredgecolor="k",
                       markersize=7),
            ],
            title="Net Regulation", loc="upper left",
            bbox_to_anchor=(1.03, 1.0), frameon=True,
            prop={"size": 6, "weight": "bold"},
            title_fontproperties={"size": 7, "weight": "bold"},
            handletextpad=1.0, labelspacing=0.4)
        ax.add_artist(reg_legend)

        # FDR size legend
        fdrs_sorted = np.sort(plot_df["-log10(FDR)"].astype(float).values)
        n_ticks = 3
        if len(fdrs_sorted) >= n_ticks:
            percentiles = np.linspace(0, 100, n_ticks + 2)[1:-1]
            # Convert -log10(FDR) back to FDR for the legend labels
            fdr_ticks = sorted(
                set(10**-(float(np.percentile(fdrs_sorted, p)))
                    for p in percentiles),
                reverse=True)
        else:
            fdr_ticks = sorted((10**-fdrs_sorted).tolist(), reverse=True)

        size_handles = []
        for fdr_val in fdr_ticks:
            neg_log = -np.log10(max(fdr_val, 1e-300))
            sz = scale_sizes(np.array([neg_log]),
                             vmin=neg_log_fdr.min(),
                             vmax=neg_log_fdr.max(),
                             size_min=BUBBLE_MIN_SIZE,
                             size_max=BUBBLE_MAX_SIZE)[0]
            lbl = (f"{fdr_val:.1e}" if fdr_val < 0.001
                   else f"{fdr_val:.4f}")
            size_handles.append(
                plt.scatter([], [], s=sz, color="gray", edgecolor="k",
                            alpha=0.8, label=lbl))

        ax.legend(handles=size_handles, title="Net FDR",
                  loc="upper left", bbox_to_anchor=(1.03, 0.5),
                  frameon=True,
                  prop={"size": 6, "weight": "bold"},
                  title_fontproperties={"size": 7, "weight": "bold"},
                  handletextpad=1.0, labelspacing=0.4)

        plt.tight_layout()

        # Reattach missing context columns for the exported CSV
        plot_df["Level"]      = level
        plot_df["Ref"]        = ref_label
        plot_df["Comparison"] = comparison
        plot_df["GeneSet"]    = gs

        # Save
        bubble_dir = os.path.join(OUT_BASE, level, ref_label,
                                  comparison, "bubble_plots")
        os.makedirs(bubble_dir, exist_ok=True)
        csv_out = os.path.join(bubble_dir, f"bubble_data_Net_Regulated_{gs}.csv")
        
        # Clean column order for CSV export
        plot_df = plot_df[["CellType", "Term", "Regulation", "Net_Signed_Sig", "-log10(FDR)", "Level", "Ref", "Comparison", "GeneSet"]]
        plot_df.to_csv(csv_out, index=False)
        
        pdf_out = os.path.join(bubble_dir, f"bubble_plot_{gs}.pdf")
        plt.savefig(pdf_out, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {pdf_out}")

print("\nAll steps complete!")


STEP 3: Cross-cell-type bubble plots
  Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_state/Ref_3month/6month_protocol_vs_3month-protocol/bubble_plots/bubble_plot_MSigDB_Hallmark_2020.pdf
  Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_state/Ref_3month/PCMV_PRV_vs_3month-protocol/bubble_plots/bubble_plot_MSigDB_Hallmark_2020.pdf
  Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_state/Ref_3month/Prematurely_terminated_vs_3month-protocol/bubble_plots/bubble_plot_MSigDB_Hallmark_2020.pdf
  Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_state/Ref_3month/Rejection_vs_3month-protocol/bubble_plots/bubble_plot_MSigDB_Hallmark_2020.pdf
  Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_state/Ref_control/3month_

In [11]:

# Cross-celltype bubbleplots v2
print("\n" + "=" * 70)
print("STEP 3: Cross-cell-type bubble plots")
print("=" * 70)

# bubbleplot configs 
BUBBLE_WIDTH_PER_CT    = 0.01
BUBBLE_HEIGHT_PER_TERM = 0.01
BUBBLE_BASE_WIDTH      = 5
BUBBLE_BASE_HEIGHT     = 3
MIN_OVERLAP            = 2

if not all_enrichment_records:
    print("[SKIP] No enrichment records to plot.")
else:
    master_df = pd.concat(all_enrichment_records, ignore_index=True)
    pval_col = ("Adjusted P-value" if "Adjusted P-value" in master_df.columns
                else "FDR")

    def scale_sizes(values, vmin, vmax, size_min, size_max):
        if vmax == vmin:
            return np.full(len(values), (size_min + size_max) / 2)
        normalized = (values - vmin) / (vmax - vmin)
        return size_min + normalized * (size_max - size_min)

    for (level, ref_label, comparison, gs), grp in master_df.groupby(
        ["Level", "Ref", "Comparison", "GeneSet"]
    ):
        if grp.empty:
            continue

        grp = grp.copy()
        grp["-log10(FDR)"] = -np.log10(
            grp[pval_col].astype(float).clip(lower=1e-300))

        # Overlapping pathways (shared by >= MIN_OVERLAP cell types)
        term_overlap = grp.groupby("Term")["CellType"].nunique()
        overlapping  = term_overlap[term_overlap >= MIN_OVERLAP]
        if overlapping.empty:
            continue

        # Rank: most shared first, then by mean significance
        term_sig = grp.groupby("Term")["-log10(FDR)"].mean()
        rank_df = (
            pd.DataFrame({"Term": overlapping.index,
                           "n_ct": overlapping.values})
            .merge(term_sig.rename("avg_sig"),
                   left_on="Term", right_index=True)
            .sort_values(["n_ct", "avg_sig"], ascending=[False, False])
        )
        top_terms = rank_df["Term"].head(TOP_N_BUBBLE).tolist()
        plot_df = grp[grp["Term"].isin(top_terms)].copy()
        if plot_df.empty:
            continue

        # Sort: Up-regulated on top, Down on bottom
        plot_df["Sign"] = plot_df["Regulation"].map(
            {"Up-regulated": 1, "Down-regulated": -1})
        term_stats = (
            plot_df.groupby("Term")
            .agg(mean_sign=("Sign", "mean"),
                 mean_sig=("-log10(FDR)", "mean"))
            .sort_values(["mean_sign", "mean_sig"], ascending=[True, True])
        )
        sorted_terms = term_stats.index.tolist()
        plot_df["Term"] = pd.Categorical(
            plot_df["Term"], categories=sorted_terms, ordered=True)
        plot_df = plot_df.sort_values(["Term", "CellType"])

        # Dot colors
        plot_df["DotColor"] = plot_df["Regulation"].map(
            {"Up-regulated": UP_COLOR, "Down-regulated": DOWN_COLOR}
        ).fillna("lightgray")

        # Dot sizes
        neg_log_fdr = plot_df["-log10(FDR)"].values
        dot_sizes = scale_sizes(neg_log_fdr,
                                vmin=neg_log_fdr.min(),
                                vmax=neg_log_fdr.max(),
                                size_min=BUBBLE_MIN_SIZE,
                                size_max=BUBBLE_MAX_SIZE)

        # Figure dimensions
        num_ct    = plot_df["CellType"].nunique()
        num_terms = len(sorted_terms)
        fig_w = BUBBLE_BASE_WIDTH  + num_ct    * BUBBLE_WIDTH_PER_CT
        fig_h = BUBBLE_BASE_HEIGHT + num_terms * BUBBLE_HEIGHT_PER_TERM

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))

        ax.scatter(plot_df["CellType"], plot_df["Term"],
                   s=dot_sizes, c=plot_df["DotColor"],
                   edgecolor="black", linewidth=0.5,
                   alpha=0.93, zorder=3)

        ax.grid(True, linestyle="--", linewidth=0.5,
                color="gray", alpha=0.5, zorder=0)
        ax.margins(x=0.15, y=0.15)
        ax.set_xlabel("Cell Type", fontsize=7, fontweight="bold")
        ax.set_ylabel("")
        ax.set_title(
            f"Bubble plot: {comparison.replace('_', ' ')}\n"
            f"(Top {TOP_N_BUBBLE} overlapping, {gs})",
            fontsize=8, fontweight="bold", pad=8)
        plt.xticks(rotation=90, ha="center", fontsize=6, fontweight="bold")
        plt.yticks(fontsize=7, fontweight="bold")

        # Regulation legend
        reg_legend = ax.legend(
            handles=[
                Line2D([0], [0], marker="o", color="w", label="Up-regulated",
                       markerfacecolor=UP_COLOR, markeredgecolor="k",
                       markersize=7),
                Line2D([0], [0], marker="o", color="w", label="Down-regulated",
                       markerfacecolor=DOWN_COLOR, markeredgecolor="k",
                       markersize=7),
            ],
            title="Regulation", loc="upper left",
            bbox_to_anchor=(1.03, 1.0), frameon=True,
            prop={"size": 6, "weight": "bold"},
            title_fontproperties={"size": 7, "weight": "bold"},
            handletextpad=1.0, labelspacing=0.4)
        ax.add_artist(reg_legend)

        # FDR size legend
        fdrs_sorted = np.sort(plot_df[pval_col].astype(float).values)
        n_ticks = 3
        if len(fdrs_sorted) >= n_ticks:
            percentiles = np.linspace(0, 100, n_ticks + 2)[1:-1]
            fdr_ticks = sorted(
                set(float(np.percentile(fdrs_sorted, p))
                    for p in percentiles),
                reverse=True)
        else:
            fdr_ticks = sorted(fdrs_sorted.tolist(), reverse=True)

        size_handles = []
        for fdr_val in fdr_ticks:
            neg_log = -np.log10(max(fdr_val, 1e-300))
            sz = scale_sizes(np.array([neg_log]),
                             vmin=neg_log_fdr.min(),
                             vmax=neg_log_fdr.max(),
                             size_min=BUBBLE_MIN_SIZE,
                             size_max=BUBBLE_MAX_SIZE)[0]
            lbl = (f"{fdr_val:.1e}" if fdr_val < 0.001
                   else f"{fdr_val:.4f}")
            size_handles.append(
                plt.scatter([], [], s=sz, color="gray", edgecolor="k",
                            alpha=0.8, label=lbl))

        ax.legend(handles=size_handles, title="FDR",
                  loc="upper left", bbox_to_anchor=(1.03, 0.5),
                  frameon=True,
                  prop={"size": 6, "weight": "bold"},
                  title_fontproperties={"size": 7, "weight": "bold"},
                  handletextpad=1.0, labelspacing=0.4)

        plt.tight_layout()

        # Save
        bubble_dir = os.path.join(OUT_BASE, level, ref_label,
                                  comparison, "bubble_plots")
        os.makedirs(bubble_dir, exist_ok=True)
        csv_out = os.path.join(bubble_dir, f"bubble_data_{gs}.csv")
        plot_df.to_csv(csv_out, index=False)
        pdf_out = os.path.join(bubble_dir, f"bubble_plot_{gs}.pdf")
        plt.savefig(pdf_out, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {pdf_out}")

print("\nAll steps complete!")


STEP 3: Cross-cell-type bubble plots
  Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_state/Ref_3month/6month_protocol_vs_3month-protocol/bubble_plots/bubble_plot_MSigDB_Hallmark_2020.pdf
  Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_state/Ref_3month/PCMV_PRV_vs_3month-protocol/bubble_plots/bubble_plot_MSigDB_Hallmark_2020.pdf
  Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_state/Ref_3month/Prematurely_terminated_vs_3month-protocol/bubble_plots/bubble_plot_MSigDB_Hallmark_2020.pdf
  Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_state/Ref_3month/Rejection_vs_3month-protocol/bubble_plots/bubble_plot_MSigDB_Hallmark_2020.pdf
  Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pathway_enrichment/v2/cell_state/Ref_control/3month_